 # Section 1: Setup, Unified Paths & Global Settings

 This cell loads essential utilities, configures global Pandas display parameters,

 silences silent-downcasting warnings, and dynamically maps project directories.

In [1]:
# 3. Path Selection Toggle

# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR
USE_LOCAL_DATA = True  # Set False for GLOBAL_DATA_DIR

In [2]:
%load_ext autoreload
%autoreload 2

import ast
from pathlib import Path

import numpy as np
import pandas as pd

from core.contracts import EngineInput, FilterPack
from core.paths import (
    GLOBAL_DATA_DIR,
    GLOBAL_PROCESSED_DIR,
    LOCAL_DATA_DIR,
    NOTEBOOKS_RLVR_ROOT,
    OUTPUT_DIR,
    PROJECT_ROOT,
)
from core.settings import TradingConfig
from core.utils import SystemUtils as SU
from data_pipeline import generate_features
from walk_forward import (
    AlphaEngine,
    create_walk_forward_analyzer,
    run_headless_simulation,
)

# 1. Global Configuration Options
config = TradingConfig()

# 2. Display Settings & Silence Warnings
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 3000)

# 3. Path Selection Toggle
# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR

data_dir = LOCAL_DATA_DIR if USE_LOCAL_DATA else GLOBAL_DATA_DIR
processed_dir = GLOBAL_PROCESSED_DIR
output_dir = OUTPUT_DIR

# 4. Define Target File Paths
ohlcv_filename = (
    "df_ohlcv.parquet" if USE_LOCAL_DATA else "df_OHLCV_stocks_etfs.parquet"
)
fed_filename = (
    "df_fed.parquet"
    if USE_LOCAL_DATA
    else "High_Yield_Spread_T10Y2Y_Spread.parquet"
)

raw_paths = {
    "df_ohlcv": data_dir / ohlcv_filename,
    "df_fed": data_dir / fed_filename,
    "df_indices": data_dir / "df_indices.parquet",
}

features_df_path = data_dir / "features_df.parquet"
macro_df_path = data_dir / "macro_df.parquet"

print(
    f"Paths Resolved:\n"
    f" - Active Mode:          {'LOCAL' if USE_LOCAL_DATA else 'GLOBAL'}\n"
    f" - Active Data Dir:       {data_dir}\n"
    f" - Global Processed Dir: {GLOBAL_PROCESSED_DIR}\n"
    f" - Output Dir:           {OUTPUT_DIR}\n"
)

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

Paths Resolved:
 - Active Mode:          LOCAL
 - Active Data Dir:       C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
 - Global Processed Dir: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
 - Output Dir:           C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



# Section 2: Ingest Datasets & Generate Features

Loads raw input files and retrieves pre-calculated features or runs feature generation.

In [3]:
# Load Raw Files
raw_dfs = {}

for name, path in raw_paths.items():
    if path.exists():
        raw_dfs[name] = pd.read_parquet(path)
        print(f"[OK] Successfully loaded {name:<10} from {path.name}")
    else:
        print(f"[ERROR] File not found: {path}")

df_ohlcv = raw_dfs.get("df_ohlcv")
df_fed = raw_dfs.get("df_fed")
df_indices = raw_dfs.get("df_indices")

[OK] Successfully loaded df_ohlcv   from df_ohlcv.parquet
[OK] Successfully loaded df_fed     from df_fed.parquet
[OK] Successfully loaded df_indices from df_indices.parquet


In [4]:
# Load Processed Features or Generate If Missing / Running Global
if USE_LOCAL_DATA and features_df_path.exists() and macro_df_path.exists():
    features_df = pd.read_parquet(features_df_path)
    macro_df = pd.read_parquet(macro_df_path)
    print(f"[OK] Loaded features_df from local disk | Shape: {features_df.shape}")
    print(f"[OK] Loaded macro_df from local disk    | Shape: {macro_df.shape}")
else:
    reason = (
        "USE_LOCAL_DATA is False"
        if not USE_LOCAL_DATA
        else "Local feature files missing"
    )
    print(f"[INFO] {reason}. Generating features (~6 mins)...")

    features_df, macro_df = generate_features(
        df_ohlcv=df_ohlcv, df_indices=df_indices, df_fed=df_fed, config=config
    )
    print(f"[OK] Generated features_df  | Shape: {features_df.shape}")
    print(f"[OK] Generated macro_df     | Shape: {macro_df.shape}")

[OK] Loaded features_df from local disk | Shape: (8542534, 21)
[OK] Loaded macro_df from local disk    | Shape: (7466, 10)


In [5]:
# Cache Processed DataFrames to GLOBAL_PROCESSED_DIR when running Global
if not USE_LOCAL_DATA:
    GLOBAL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Saving DataFrames to: {GLOBAL_PROCESSED_DIR}\n")

    dfs_to_save = {
        "df_ohlcv.parquet": df_ohlcv,
        "df_indices.parquet": df_indices,
        "df_fed.parquet": df_fed,
        "features_df.parquet": features_df,
        "macro_df.parquet": macro_df,
    }

    for filename, df in dfs_to_save.items():
        if df is not None:
            save_path = GLOBAL_PROCESSED_DIR / filename
            df.to_parquet(save_path)
            print(f"[OK] Saved {filename:<20} | Shape: {df.shape}")
        else:
            print(f"[SKIP] {filename:<19} | DataFrame missing in memory")

In [6]:
###############################
###############################

In [7]:
print("🚀 Unstacking Wide Matrices...")

# 1. Unstack directly.
# (Since df_ohlcv is already a perfect grid of master_dates x tickers,
# unstacking automatically creates perfectly aligned wide dataframes).
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0)
df_trp_wide = features_df["TRP"].unstack(level=0)

print("Applying terminal NaN replacements for model inputs...")

# 2. Terminal Fills ONLY.
# Note: 0-to-NaN conversion and bounded ffills were ALREADY applied upstream to df_ohlcv.
# Applying ffill here would incorrectly double your max_data_gap_ffill limit!
df_close_wide = df_close_wide.fillna(config.nan_price_replacement)
df_atrp_wide = df_atrp_wide.fillna(0.0)
df_trp_wide = df_trp_wide.fillna(0.0)

print("All wide dataframes unstacked and prepared for downstream modeling.")
print(
    f"[OK] Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

🚀 Unstacking Wide Matrices...
Applying terminal NaN replacements for model inputs...
All wide dataframes unstacked and prepared for downstream modeling.
[OK] Pipeline Complete. Tickers: 1708, Days: 7466


 # Section 3: In-Memory Structural Previews

 Outputs head and tail previews of the core dataframes.

In [8]:
# print("\n" + "=" * 50)
# print("DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)")
# print("=" * 50)

# # 1. Preview wide matrix prices
# if "df_close_wide" in locals():
#     print("\n=== df_close_wide (Prices) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(df_close_wide.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(df_close_wide.tail(3))

# # 2. Preview multi-index features
# if "features_df" in locals():
#     print("\n=== features_df (Multi-Index Features) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(features_df.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(features_df.tail(3))

# # 3. Preview macro data
# if "macro_df" in locals():
#     print("\n=== macro_df (Macro Data) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(macro_df.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(macro_df.tail(3))

 # Section 4: Engine Initialization & Stage 1 Analysis

In [9]:
# Instantiate Master Engine
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("Master AlphaEngine Ready.")

Master AlphaEngine Ready.


In [10]:
# import pandas as pd

# target_date = pd.Timestamp("2025-09-01")

# # Use level=1 for the date level (since level names aren't 'timestamp')
# dates_in_index = df_ohlcv.index.get_level_values(1).normalize().unique()
# print(f"Labor Day 2025-09-01 in index: {target_date in dates_in_index}")

# # Check if date exists (one-liner)
# any_match = (df_ohlcv.index.get_level_values(1).normalize() == target_date).any()
# print(f"Any match: {any_match}")

# # If you want to see which tickers have data on that date:
# if any_match:
#     mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
#     tickers_on_date = df_ohlcv.index.get_level_values(0)[mask].unique()
#     print(f"Tickers on 2025-09-01: {tickers_on_date}")
#     print(f"Count: {len(tickers_on_date)}")

# # Extract all data for that specific date
# if any_match:
#     # Using boolean mask (works regardless of index sorting)
#     mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
#     df_target = df_ohlcv[mask]
#     print(df_target.head())

In [ ]:
# Configuration for Interactive UI (Stage 1 Setup)
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-08-05"),
    lookback_period=189,
    holding_period=config.holding_period,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=50,  # Plot only display 50 tickers
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [12]:
# Execute Headless Agent Simulation
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df.style.format("{:.6f}"))
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
----------------------------------------------------------------------
Timeline: [2025-10-31] -> Decision: 2026-08-05 -> Entry: 2026-08-06 -> End: 2026-08-13
Selected Tickers (50):
SGOV, SHV, BIL, VBIL, MINT, USFR, BOXX, PULS, JPST, JAAA
RVMD, RY, TD, PEN, SNDK, SCHD, MU, JAZZ, IWD, ROST
NUE, HST, DELL, FNDX, HSBC, ASX, RPRX, EWT, STT, VTRS
BNS, STM, VTV, CSCO, TRGP, STX, ATI, HPE, WDC, CASY
BUD, EFV, LITE, FTNT, SN, AMAT, TRV, SCHV, SU, CSX
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.640954,0.616065,0.021100
Benchmark Gain,0.139705,0.129250,0.012054
== Gain Delta,0.501249,0.486815,0.009047
Group Sharpe,3.325074,3.276186,10.848124
Benchmark Sharpe,1.410333,1.334307,8.974761
== Sharpe Delta,1.914740,1.941879,1.873363
Group Sharpe (ATRP),0.104262,0.104903,0.136542
Benchmark Sharpe (ATRP),0.064430,0.061590,0.225413
== Sharpe (ATRP) Delta,0.039832,0.043313,-0.088871



Target Reward Signal: 0.021100


 # Section 5: Stage 2 - Cascade Filter Execution

In [13]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# LAUNCH STAGE 2 (Cascade on the Stage 1 subset)
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [14]:
# Parse and map results from Analyzer 2
result = SU.map_analyzer(analyzer2)

[SEARCH] ANALYZER SIMULATION MAP
[  0] [DATA] audit_pack (EngineOutput)
[  1]   [PLOT] portfolio_series (shape=(196,))
[  2]   [PLOT] benchmark_series (shape=(196,))
[  3]   [CALC] normalized_plot_data (shape=(196, 50))
[  4]   [FILE] tickers (len=50)
[  5]     [DOC] index_0 (str)
[  6]     [DOC] index_1 (str)
[  7]     [DOC] index_2 (str)
[  8]     [DOC] index_3 (str)
[  9]     [DOC] index_4 (str)
[ 10]     [DOC] index_5 (str)
[ 11]     [DOC] index_6 (str)
[ 12]     [DOC] index_7 (str)
[ 13]     [DOC] index_8 (str)
[ 14]     [DOC] index_9 (str)
[ 15]     [DOC] index_10 (str)
[ 16]     [DOC] index_11 (str)
[ 17]     [DOC] index_12 (str)
[ 18]     [DOC] index_13 (str)
[ 19]     [DOC] index_14 (str)
[ 20]     [DOC] index_15 (str)
[ 21]     [DOC] index_16 (str)
[ 22]     [DOC] index_17 (str)
[ 23]     [DOC] index_18 (str)
[ 24]     [DOC] index_19 (str)
[ 25]     [DOC] index_20 (str)
[ 26]     [DOC] index_21 (str)
[ 27]     [DOC] index_22 (str)
[ 28]     [DOC] index_23 (str)
[ 29]     [DOC

In [15]:
# Test different methods to extract resulting target tickers
_tickers = SU.fetch(4, result)
print(f"_tickers (by index): {_tickers}\n")

_tickers = SU.fetch("tickers", result)
print(f"_tickers (by key): {_tickers}\n")

print(f'result[4]["obj"]: {result[4]["obj"]}')

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'VBIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'JAAA',
 'RVMD',
 'RY',
 'TD',
 'PEN',
 'SNDK',
 'SCHD',
 'MU',
 'JAZZ',
 'IWD',
 'ROST',
 'NUE',
 'HST',
 'DELL',
 'FNDX',
 'HSBC',
 'ASX',
 'RPRX',
 'EWT',
 'STT',
 'VTRS',
 'BNS',
 'STM',
 'VTV',
 'CSCO',
 'TRGP',
 'STX',
 'ATI',
 'HPE',
 'WDC',
 'CASY',
 'BUD',
 'EFV',
 'LITE',
 'FTNT',
 'SN',
 'AMAT',
 'TRV',
 'SCHV',
 'SU',
 'CSX']

_tickers (by index): ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'VBIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'JAAA',
 'RVMD',
 'RY',
 'TD',
 'PEN',
 'SNDK',
 'SCHD',
 'MU',
 'JAZZ',
 'IWD',
 'ROST',
 'NUE',
 'HST',
 'DELL',
 'FNDX',
 'HSBC',
 'ASX',
 'RPRX',
 'EWT',
 'STT',
 'VTRS',
 'BNS',
 'STM',
 'VTV',
 'CSCO',
 'TRGP',
 'STX',
 'ATI',
 'HPE',
 'WDC',
 'CASY',
 'BUD',
 'EFV',
 'LITE',
 'FTNT',
 'SN',
 'AMAT',
 'TRV',
 'SCHV',
 'SU',
 'CSX']

_tickers (by key): ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']

result[4]["obj"]: ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']


 # Section 6: Verification, Exports & Auditing

In [16]:
# Export analytical trace logs to Excel
SU.export_audit_to_excel(
    audit_pack=analyzer2.last_run, filename="Audit_Verification_Report.xlsx"
)

[FILE] [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx
[DONE] Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/Audit_Verification_Report.xlsx')

In [17]:
# Export last-run stacked OHLCV ticker records
SU.export_last_run_tickers_data_to_csv(
    analyzer=analyzer2,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename="last_run_tickers_stacked.csv",
)

Creating combined dictionary for 51 ticker(s)
Date range: 2025-10-31 00:00:00 to 2026-08-13 00:00:00
Data retrieved for 51 ticker(s) from 2025-10-31 00:00:00 to 2026-08-13 00:00:00
Total rows: 9996
Date range in data: 2025-10-31 00:00:00 to 2026-08-13 00:00:00
  SGOV: 196 rows
  SHV: 196 rows
  BIL: 196 rows
  VBIL: 196 rows
  MINT: 196 rows
  USFR: 196 rows
  BOXX: 196 rows
  PULS: 196 rows
  JPST: 196 rows
  JAAA: 196 rows
  RVMD: 196 rows
  RY: 196 rows
  TD: 196 rows
  PEN: 196 rows
  SNDK: 196 rows
  SCHD: 196 rows
  MU: 196 rows
  JAZZ: 196 rows
  IWD: 196 rows
  ROST: 196 rows
  NUE: 196 rows
  HST: 196 rows
  DELL: 196 rows
  FNDX: 196 rows
  HSBC: 196 rows
  ASX: 196 rows
  RPRX: 196 rows
  EWT: 196 rows
  STT: 196 rows
  VTRS: 196 rows
  BNS: 196 rows
  STM: 196 rows
  VTV: 196 rows
  CSCO: 196 rows
  TRGP: 196 rows
  STX: 196 rows
  ATI: 196 rows
  HPE: 196 rows
  WDC: 196 rows
  CASY: 196 rows
  BUD: 196 rows
  EFV: 196 rows
  LITE: 196 rows
  FTNT: 196 rows
  SN: 196 rows


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/last_run_tickers_stacked.csv')

 # Section 7: Slicing Tests & Dictionary Parsing

In [18]:
# Verify captured ticker names
print(f"_tickers: {_tickers}")

_tickers: ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']


In [19]:
# Test various pandas multi-index slicing configurations
##############
test_tickers = ["NVDA", "GOOG"]
##############

idx = pd.IndexSlice
columns = ["Adj Close", "Volume"]

# # Slice 1: All dates, all columns
# display(df_ohlcv.loc[idx[test_tickers]].copy().head(3))

# # Slice 2: specific start date, all columns
# display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], :].head(3))

# # Slice 3: specific start date, selected columns
# display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], columns].head(3))

# Slice 4: specific start/end range, all columns
display(
    df_ohlcv.loc[
        idx[test_tickers, pd.Timestamp("2026-08-07") : pd.Timestamp("2026-08-14")], :
    ]
)

Adj Open  Adj High  Adj Low  Adj Close       Volume
Ticker Date                                                           
NVDA   2026-08-07   221.540   224.760  220.660    223.960  105669400.0
       2026-08-10   223.400   224.140  216.770    217.550  115846600.0
       2026-08-11   222.170   222.200  216.200    217.500  101273100.0
       2026-08-12   221.040   225.100  220.200    224.090  108783600.0
       2026-08-13   225.060   227.230  223.710    225.300   98867200.0
       2026-08-14   226.770   227.490  224.500    225.160   75680900.0
GOOG   2026-08-07   355.049   357.318  352.591    353.241   14526124.0
       2026-08-10   354.010   355.699  351.242    355.609   12797803.0
       2026-08-11   353.770   354.770  342.348    342.777   18402139.0
       2026-08-12   345.106   345.276  339.600    342.148   15618933.0
       2026-08-13   344.346   345.606  342.178    343.717   15007536.0
       2026-08-14   344.037   347.834  341.848    343.317   10953907.0

In [20]:
# Test custom combined dictionary extraction
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=_tickers,
    date_start="2026-05-01",
    date_end=None,
    verbose=False,
)

{'SGOV':             Adj Open  Adj High   Adj Low  Adj Close      Volume       ATR      ATRP       TRP       RSI    Mom_21    Mom_63   Mom_126  Mom_252_21  Consistency     IR_63   Beta_63   IVol_63  SemiDev_63  Trend_R2_63  DD_21    Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount  RecentStaleDays  IsZeroPrice
 Date                                                                                                                                                                                                                                                                                                                        
 2026-05-01   99.2097   99.2097   99.1998    99.2097  34218740.0  0.019252  0.000194  0.000378  0.999997  0.003169  0.009115  0.018944    0.037155          0.6 -0.061535 -0.003311  0.000116         0.0     0.999421    0.0  0.000378         0.011905      1.368668e+09                  0.0              0.0            0
 2026-05-04   99.2196   99.2196   99.2

 # Section 8: Finviz Ingestion & Filtering

In [21]:
# Load latest updated Finviz dataset dynamically
def load_latest_finviz_parquet(data_dir: Path) -> pd.DataFrame:
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"
    files = list(data_dir.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {data_dir.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Load the file using the unified data_dir path
df_finviz = load_latest_finviz_parquet(GLOBAL_DATA_DIR)
print(
    f"df_finviz raw records shape: {df_finviz.shape if df_finviz is not None else None}"
)

DEBUG: Loading file: 2026-09-04_df_finviz_merged_stocks_etfs.parquet
df_finviz raw records shape: (1435, 139)


In [22]:
display(df_finviz.head())
display(df_finviz.tail())

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
NVDA,1,NVIDIA Corp,"DJIA, NDX, S&P 500",Technology,Semiconductors,USA,NASD,"Technology, Semiconductors",5551680.0,1,5551680.0,29.12,NaN,0.23,18.32,24.29,55.87,43.71,9.48,4.12,0.32,0.28,9/10/2026,0.82,7.91,2.47,NaN,NaN,NaN,NaN,NaN,105.85,128.21,124.42,83.38,302970.0,192880.0,6.22,4.28,24150.0,23180.0,96.01,3.80,-0.51,72.32,0.40,1.23,2.08,285.96,83.68,117.21,72.42,4.59,3.85,0.16,0.17,74.67,65.21,63.66,5.941869,5.89,5.19,12.32,29.55,34.20,23.52,2.22,7.38,3.203681,2.78,2.61,4.67,9.40,17.12,-0.05,21.37,-2.61,40.40,164.07 - 236.54,-2.61,690979.97,60.39,Aug 26/a,1/22/1999,Yes,Yes,42000.0,-0.32,1.16,1.21,137.40,0.99,135352420,334.32,228.45,231.09,234.76,229.63,230.36,0.84,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,30.321863,16498.527747,1470.805672,8.528811,22.147372,3.790307,4.943420,10.829629,2.35050,0.974388,1.832418,1.182775,3.317754,6.114338,1.745815,1.325857,2.142471,1.236037,1.336170,2.097342,1.239974,0.881853,1.327768,1.152853
AAPL,2,Apple Inc,"DJIA, NDX, S&P 500",Technology,Consumer Electronics,USA,NASD,"Technology, Consumer Electronics",4669700.0,2,4669700.0,36.68,NaN,2.64,10.00,43.47,74.84,34.16,7.36,4.28,0.34,1.06,8/10/2026,13.66,8.72,1.98,NaN,NaN,NaN,NaN,NaN,16.36,29.13,32.57,14.24,466820.0,128930.0,6.77,0.35,14610.0,14580.0,99.78,0.12,-2.25,68.91,0.16,0.80,2.14,116.33,36.08,148.75,72.08,1.00,0.93,0.66,0.78,48.65,33.17,27.62,-1.587057,0.08,2.42,4.11,24.28,33.44,17.70,1.07,7.64,2.387724,2.75,2.06,2.18,1.53,12.72,-7.14,16.88,-7.14,41.61,225.95 - 344.57,-7.14,502870.89,53.89,Jul 30/a,12/12/1980,Yes,Yes,166000.0,-2.54,0.03,2.16,54.40,0.73,39606884,332.70,328.21,328.30,328.93,317.86,319.97,-2.51,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-4.930988,-6.851985,0.389577,1.822393,3.099619,1.380395,3.543168,5.917134,1.82194,3.501151,5.852721,1.745992,-1.461901,-1.708463,0.759624,1.105850,1.519829,1.217583,1.764223,2.549874,1.356733,1.236677,1.800468,1.244853
GOOG,4,Alphabet Inc,"NDX, S&P 500",Communication Services,Internet Content & Information,USA,NASD,"Communication Services, Internet Content & Inf...",4121960.0,3,4121960.0,16.85,NaN,1.22,9.24,6.59,17.00,77.37,50.90,19.72,0.24,0.86,9/4/2026,7.68,19.91,3.02,NaN,NaN,NaN,NaN,NaN,23.98,294.02,112.71,20.23,446310.0,244120.0,216.66,2.89,5530.0,5140.0,92.94,58.00,-0.01,28.69,2.19,0.83,2.09,42.89,34.30,48.68,32.33,2.72,2.64,0.18,0.18,60.94,33.96,54.70,1.053298,-2.21,-5.98,-8.33,12.41,44.12,6.85,1.20,7.83,2.335153,1.79,1.73,-1.62,-3.24,0.16,-12.18,6.48,-17.10,47.87,226.76 - 404.47,-17.10,1279.24,43.97,Jul 22/a,3/27/2014,Yes,Yes,190820.0,-1.14,

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
NEAR,395,iShares Short Duration Bond Active ETF,-,Financial,Exchange Traded Fund,USA,CBOE,"Financial, Exchange Traded Fund, Bonds - Broad...",5170.0,1555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.44,2.24,9/1/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.80,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.099443,-0.38,-0.23,-0.32,-1.29,-1.62,-1.51,0.03,0.07,0.139082,0.12,0.11,-0.33,-0.34,-0.99,-0.77,0.16,-2.02,0.16,50.25 - 51.37,-2.09,14.46,34.86,-,9/26/2013,Yes,Yes,NaN,0.02,-0.02,NaN,0.57146,0.80,457427,NaN,50.33,50.32,50.34,50.31,50.33,0.00,Bonds - Broad Market,Bonds,0.25,1875.0,5170.0,NaN,4.76,545.36,11.80,1410.00,37.47,2.80,5.34,3.96,"U.S., fixed-income, bonds, investment-grade",2.261030,5.662687,1.504472,-4.125219,-4.859644,0.448359,-5.335042,-5.826272,0.389819,-4.800783,-5.246555,0.417803,0.213694,0.281891,1.036578,-1.638454,-1.989346,0.761427,-1.241620,-1.571264,0.815035,-1.020340,-1.317536,0.845299
ISTB,397,iShares Core 1-5 Year USD Bond ETF,-,Financial,Exchange Traded Fund,USA,NASD,"Financial, Exchange Traded Fund, Bonds - Broad...",5120.0,1557,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.31,2.06,9/1/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.36,0.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.125681,-0.40,-0.27,-0.50,-1.77,-1.97,-1.95,0.11,0.08,0.167364,0.09,0.10,-0.40,-0.47,-1.30,-1.08,0.14,-2.55,0.14,47.73 - 49.05,-7.15,6.03,36.77,-,10/22/2012,Yes,Yes,NaN,0.02,-0.02,NaN,0.43804,0.63,277711,NaN,47.80,47.79,47.81,47.78,47.80,0.00,Bonds - Broad Market,Bonds,0.06,7403.0,5120.0,NaN,0.37,235.19,4.82,515.15,11.19,2.28,4.99,1.92,"U.S., fixed-income, bonds",5.550719,21.961187,2.956458,-3.957695,-4.651757,0.457350,-4.274781,-4.912347,0.458738,-4.151985,-4.674508,0.479476,-0.499753,-0.666966,0.923388,-1.607871,-2.028171,0.781120,-1.386290,-1.762402,0.806722,-1.084550,-1.400317,0.841349
EMLC,398,VanEck J.P. Morgan EM Local Currency Bond ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Bonds - Treas...",5110.0,1558,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.18,1.58,9/1/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.90,3.93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.786473,0.23,0.31,1.91,0.16,1.83,-0.74,0.37,0.11,0.429185,0.23,0.27,-0.16,0.38,-0.18,-1.10,2.07,-3.76,3.35,24.80 - 26.63,-54.17,15.17,50.73,-,7/23/2010,Yes,Yes,NaN,-0.08,0.08,NaN,2.07000,1.15,2390537,NaN,25.63,25.65,25.68,25.63,25.63,0.00,Bonds - Tre

In [23]:
raw_text = """
SMTC, TER, MTSI, PRAX, TTMI, LSCC,
"""

# Split by comma, strip whitespace, and filter out any empty strings
ticker_list = [item.strip() for item in raw_text.split(",") if item.strip()]

print(ticker_list)

['SMTC', 'TER', 'MTSI', 'PRAX', 'TTMI', 'LSCC']


In [24]:
# Set test target tickers
##################
# target_tickers = ["GLW", "NOK", "DELL", "NVDA", "GOOG"]
target_tickers = ticker_list

print(f"target_tickers: {target_tickers}")
##################

target_tickers: ['SMTC', 'TER', 'MTSI', 'PRAX', 'TTMI', 'LSCC']


In [25]:
# Clean index extraction using ast exception parsing if tickers are missing
try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Strip out the missing symbols and retry slicing
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz].copy()
    result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

print(f"result shape: {result.shape}")

Missing tickers: ['PRAX']
result shape: (5, 139)


In [26]:
# Display filtered Finviz information and schema details
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, TER to TTMI
Columns: 139 entries, No. to Omega 250d
dtypes: float64(120), int64(3), object(16)
memory usage: 5.5+ KB


In [27]:
# Check specific target column (Market Capitalization)
result["MktCap AUM, M"]

TER     55820.0
MTSI    20540.0
LSCC    16460.0
SMTC    13800.0
TTMI    13230.0
Name: MktCap AUM, M, dtype: float64

In [28]:
# Sort records descendingly by Market Capitalization
result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

In [29]:
# View final sorted DataFrame
print(result)

      No.                                  Company         Index      Sector                             Industry Country Exchange                                             Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E   PEG    P/S    P/B     P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %   EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %  ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W 

In [30]:
# # Export processed dataframe output directly to trash file
# result.to_csv(output_dir / "_trash.csv")